In [1]:
import pandas as pd
import json
import numpy as np

In [85]:
_k = 10
_ret = 'e5'
_task = 'dl'
target_metric = 'f1'

if(_task=='nq'):
    _dataset_dev, _dataset_test, _prefix, _suffix = 'nq_dev', ['nq_test'], 'short', 'concise'
elif(_task=='dl'):
    _dataset_dev, _dataset_test, _prefix, _suffix = 'dev_small', ['19', '20'], 'random', 'prompt1'

f = open(f'../../rag_utility/eval_results/{_prefix}_answers_0shot_1calls_0_0_bm25_dl_{_dataset_dev}_{_suffix}_eval.json')
zero_evals = json.load(f)
f.close()

f = open(f'../../rag_utility/eval_results/{_prefix}_answers_{_k}shot_1calls_1_0_{_ret}_dl_{_dataset_dev}_{_suffix}_eval.json')
k_evals = json.load(f)
f.close()

f = open(f'../../rag_utility/gen_results/{_prefix}_answers_{_k}shot_1calls_1_0_{_ret}_dl_{_dataset_dev}_{_suffix}.json')
k_gens = json.load(f)
f.close()

f = open(f'../coherence_eval/log_prob_temp_res/full_context/{_dataset_dev}_{_ret}_{_k}.json')
perpC = json.load(f)
f.close()

qpp_df = pd.read_csv(f'./precomputed_qpps/{_ret}_{_k}_combined_qpp_{_dataset_dev}.csv')

dev_res = qpp_df[['qid', 'query']].drop_duplicates().copy()

# Expand the dataframe for the convenience of analysis
for qpp_name in qpp_df.qpp_method.unique():
    value_dict = dict(zip(qpp_df[qpp_df.qpp_method==qpp_name]['qid'], qpp_df[qpp_df.qpp_method==qpp_name]['qpp_estimate']))
    dev_res[qpp_name] = dev_res.qid.apply(lambda _qid: value_dict[_qid])

if(_task=='nq'):
    base_f1_dict = {item[0]: item[1]['0']['0']['F1'] for item in zero_evals.items()}
    f1_dict = {item[0]: item[1]['0']['0']['F1'] for item in k_evals.items()}
    # em_dict = {item[0]: item[1]['0']['0']['EM'] for item in k_evals.items()}
    kshot_prob_dict = {item[0]: np.mean(eval(item[1]['0']['0']['probs'])) for item in k_gens.items()}
elif(_task=='dl'):
    base_f1_dict = {item[0]: item[1]['0']['0']['qrel_1']['f1']['max'] for item in zero_evals.items() if ('0' in item[1].keys())}
    f1_dict = {item[0]: item[1]['0']['0']['qrel_1']['f1']['max'] for item in k_evals.items() if ('0' in item[1].keys())}
    kshot_prob_dict = {item[0]: np.mean(eval(item[1]['0']['0']['probs'])) for item in k_gens.items() if ('0' in item[1].keys())}

dev_res = dev_res[dev_res.qid.astype('str').isin(f1_dict.keys())]
dev_res['f1'] = dev_res.qid.apply(lambda x: f1_dict[str(x)])
dev_res['utility'] = dev_res.qid.apply(lambda x: f1_dict[str(x)]-base_f1_dict[str(x)])
# dev_res['em'] = dev_res.qid.apply(lambda x: em_dict[str(x)])
dev_res['prob(k)'] = dev_res.qid.apply(lambda x: kshot_prob_dict[str(x)])
dev_res['perpC'] = dev_res.qid.apply(lambda x: perpC[str(x)])

dev_res = dev_res.dropna(axis='index')
dev_res.head(3)

,qid,query,nqc,maxScore,spatial,a_ratio,bertQPP,bertQPP(QV),f1,utility,prob(k),perpC
0,1000000,where does real insulin come from,0.000254,0.891078,2555.967041,1.028092,0.067983,0.036887,0.551614,-0.004040,-0.126748,-1.867188
1,1000004,where does name nora come from,0.000545,0.904835,2653.737305,1.051400,0.335996,0.243154,0.587759,0.023156,-0.394669,-1.490234
2,1000006,where does most of the iron ore come from,0.000047,0.873369,2456.736084,1.034723,0.061252,0.115092,0.514067,-0.058122,-0.186950,-1.430664


In [86]:
dev_res.shape

(6980, 12)

In [87]:
import numpy as np

def tool_for_aggregating_dl_performance(x):
    # the same as performLoader
    scores = []
    for answer_eval in x[1]['0'].values():
        scores.append(max(answer_eval['qrel_2']['f1']['max'], answer_eval['qrel_3']['f1']['max']))
    return np.mean(scores)

zero_evals, k_evals, k_gens = {}, {}, {}

_calls = 5 if _task=='dl' else 1

for _d in _dataset_test:
    f = open(f'../../rag_utility/eval_results/{_prefix}_answers_0shot_{_calls}calls_0_0_bm25_dl_{_d}_{_suffix}_eval.json')
    zero_evals.update(json.load(f))
    f.close()
    
    f = open(f'../../rag_utility/eval_results/{_prefix}_answers_{_k}shot_{_calls}calls_1_0_{_ret}_dl_{_d}_{_suffix}_eval.json')
    print(f'../../rag_utility/eval_results/{_prefix}_answers_{_k}shot_{_calls}calls_1_0_{_ret}_dl_{_d}_{_suffix}_eval.json')
    k_evals.update(json.load(f))
    f.close()
    
    f = open(f'../../rag_utility/gen_results/{_prefix}_answers_{_k}shot_{_calls}calls_1_0_{_ret}_dl_{_d}_{_suffix}.json')
    k_gens.update(json.load(f))
    f.close()

if(_task == 'dl'):
    _d = 'dl'
else:
    _d = 'nq_test'
    
f = open(f'../coherence_eval/log_prob_temp_res/full_context/{_d}_{_ret}_{_k}.json')
perpC.update(json.load(f))
f.close()

qpp_df = pd.read_csv(f'./precomputed_qpps/{_ret}_{_k}_combined_qpp_{_d}.csv')

test_res = qpp_df[['qid', 'query']].drop_duplicates().copy()

# Expand the dataframe for the convenience of analysis
for qpp_name in qpp_df.qpp_method.unique():
    value_dict = dict(zip(qpp_df[qpp_df.qpp_method==qpp_name]['qid'], qpp_df[qpp_df.qpp_method==qpp_name]['qpp_estimate']))
    test_res[qpp_name] = test_res.qid.apply(lambda _qid: value_dict[_qid])

if(_task=='nq'):
    base_f1_dict = {item[0]: item[1]['0']['0']['F1'] for item in zero_evals.items()}
    f1_dict = {item[0]: item[1]['0']['0']['F1'] for item in k_evals.items()}
    # em_dict = {item[0]: item[1]['0']['0']['EM'] for item in k_evals.items()}
    kshot_prob_dict = {item[0]: np.mean(eval(item[1]['0']['0']['probs'])) for item in k_gens.items()}
elif(_task=='dl'):
    base_f1_dict = {item[0]: tool_for_aggregating_dl_performance(item) for item in zero_evals.items() if ('0' in item[1].keys())}
    f1_dict = {item[0]: tool_for_aggregating_dl_performance(item) for item in k_evals.items() if ('0' in item[1].keys())}
    kshot_prob_dict = {item[0]: np.mean(eval(item[1]['0']['0']['probs'])) for item in k_gens.items() if ('0' in item[1].keys())}

test_res = test_res[test_res.qid.astype('str').isin(f1_dict.keys())]
test_res['f1'] = test_res.qid.apply(lambda x: f1_dict[str(x)])
test_res['utility'] = test_res.qid.apply(lambda x: f1_dict[str(x)]-base_f1_dict[str(x)])
# test_res['em'] = test_res.qid.apply(lambda x: em_dict[str(x)])
test_res['prob(k)'] = test_res.qid.apply(lambda x: kshot_prob_dict[str(x)])
test_res['perpC'] = test_res.qid.apply(lambda x: perpC[str(x)])

test_res.head(3)

../../rag_utility/eval_results/random_answers_10shot_5calls_1_0_e5_dl_19_prompt1_eval.json
../../rag_utility/eval_results/random_answers_10shot_5calls_1_0_e5_dl_20_prompt1_eval.json


,qid,query,nqc,maxScore,spatial,a_ratio,bertQPP,bertQPP(QV),f1,utility,prob(k),perpC
0,1030303,who is aziz hashim,0.004477,0.885742,2392.513184,1.099638,0.765038,0.770591,0.950072,0.392684,-0.062885,-1.904297
1,1037496,who is rep scalise,0.000427,0.888616,2508.355713,1.075504,0.457166,0.214858,0.725122,0.052857,-0.102809,-1.303711
2,1037798,who is robert gray,0.001273,0.860324,2308.949707,1.045476,0.319397,0.594975,0.517673,-0.039951,-0.180834,-2.052734


In [88]:
from scipy import stats

In [89]:
import pandas as pd

df_content = []

df_content.append(['nqc', stats.spearmanr(test_res.nqc, test_res[target_metric])[0], stats.kendalltau(test_res.nqc, test_res[target_metric])[0]])
df_content.append(['maxScore', stats.spearmanr(test_res.maxScore, test_res[target_metric])[0], stats.kendalltau(test_res.maxScore, test_res[target_metric])[0]])
df_content.append(['denseQPP', stats.spearmanr(test_res.spatial, test_res[target_metric])[0], stats.kendalltau(test_res.spatial, test_res[target_metric])[0]])
df_content.append(['aPairRatio', stats.spearmanr(test_res.a_ratio, test_res[target_metric])[0], stats.kendalltau(test_res.a_ratio, test_res[target_metric])[0]])
df_content.append(['bertQPP', stats.spearmanr(test_res.bertQPP, test_res[target_metric])[0], stats.kendalltau(test_res.bertQPP, test_res[target_metric])[0]])

df_content.append(['perpC', stats.spearmanr(test_res.perpC, test_res[target_metric])[0], stats.kendalltau(test_res.perpC, test_res[target_metric])[0]])

df_content.append(['prob(k)', stats.spearmanr(test_res['prob(k)'], test_res[target_metric])[0], stats.kendalltau(test_res['prob(k)'], test_res[target_metric])[0]])

a1 = pd.DataFrame(df_content, columns=['QPP_Method', 'Spearman', 'Kendall'])
a1.Spearman = a1.Spearman.apply(lambda x: round(x, 4))
a1.Kendall = a1.Kendall.apply(lambda x: round(x, 4))
a1.to_csv('./temp_for_pasting_results/a1.csv', index=False)

In [90]:
a1

,QPP_Method,Spearman,Kendall
0,nqc,-0.0039,-0.0069
1,maxScore,0.4570,0.3174
2,denseQPP,0.2422,0.1731
3,aPairRatio,-0.1145,-0.0700
4,bertQPP,0.0923,0.0601
5,perpC,0.3034,0.2003
6,prob(k),0.6186,0.4519


#### Learned Combination -> Linear Regression

In [91]:
from sklearn import linear_model
reg = linear_model.LinearRegression()

used_qpp_methods = ['nqc', 'spatial', 'maxScore', 'a_ratio', 'bertQPP']
# used_qpp_methods = ['nqc', 'maxScore', 'bertQPP']
# used_qpp_methods = ['maxScore']

retr_cen = dev_res[used_qpp_methods].apply(lambda x: np.log(1+x)).values
reader_cen = dev_res[['perpC']].values
dev_data = np.hstack((retr_cen, reader_cen))
# dev_data = retr_cen

reg.fit(dev_data, dev_res[target_metric].values)
coefs = reg.coef_
intercept = reg.intercept_
predictions = (dev_data * coefs).sum(axis=1) + intercept
predictions_with_prob = predictions + dev_res['prob(k)'].values

print(coefs)
print(intercept)
print('accuracy on Dev set', stats.spearmanr(predictions, dev_res[target_metric]))
print('accuracy on Dev set (w/prob(k))', stats.spearmanr(predictions_with_prob, dev_res[target_metric]))

try:
    f = open('./temp_for_pasting_results/weights.json', 'r+', encoding='UTF-8')
    weights = json.load(f)
    f.close()
except:
    weights = {}

weights.update({f'{_k}-{_ret}-{_task}-{target_metric}': list(coefs)})
f = open('./temp_for_pasting_results/weights.json', 'w+')
json.dump(weights, f, indent=4)
f.close()


retr_cen_test = test_res[used_qpp_methods].apply(lambda x: np.log(1+x)).values
reader_cen_test = test_res[['perpC']].values
test_data = np.hstack((retr_cen_test, reader_cen_test))
# test_data = retr_cen_test

predictions_test = ((test_data * coefs).sum(axis=1) + intercept)
predictions_test_with_prob = predictions_test + test_res['prob(k)'].values

print('accuracy on Test set', stats.spearmanr(predictions_test, test_res[target_metric])[0], stats.kendalltau(predictions_test, test_res[target_metric])[0])
print('accuracy on Test set (w/prob(k))', stats.spearmanr(predictions_test_with_prob, test_res[target_metric])[0], stats.kendalltau(predictions_test_with_prob, test_res[target_metric])[0])

[ 4.47488814e+00  7.56956925e-02  1.37045598e+00 -3.35900328e-03
  1.00491163e-01  1.99137472e-02]
-0.8587690938480421
accuracy on Dev set SignificanceResult(statistic=0.31557972085036856, pvalue=3.324197582093318e-161)
accuracy on Dev set (w/prob(k)) SignificanceResult(statistic=0.40747895380941646, pvalue=1.765859058607599e-277)
accuracy on Test set 0.4154086892488954 0.2968213058419244
accuracy on Test set (w/prob(k)) 0.6424626551651588 0.47036082474226815


#### Linear combination - minus correlation as the loss function

In [62]:
import numpy as np
from scipy.optimize import minimize

In [63]:
# X = ...  # shape (n_samples, n_models)
# y = ...  # shape (n_samples,)

used_qpp_methods = ['nqc', 'spatial', 'maxScore', 'a_ratio', 'bertQPP']
retr_cen = dev_res[used_qpp_methods].apply(lambda x: np.log(1+x)).values
reader_cen = dev_res[['perpC']].values
dev_data = np.hstack((retr_cen, reader_cen))
dev_target = dev_res[target_metric].values

def corr_loss(v):
    w = np.exp(v) / np.sum(np.exp(v))
    yhat = dev_data.dot(w)
    # print(yhat)
    r = np.corrcoef(yhat, dev_target)[0,1]  # Pearson's r from numpy
    # print(r)
    rho = stats.spearmanr(yhat, dev_target)[0]
    # print(rho)
    return -r

In [76]:
v0 = np.zeros(dev_data.shape[1])
# v0 = coefs.copy()
res = minimize(corr_loss, v0, method='BFGS',
               options={'gtol':1e-6, 'maxiter':500})
w_opt = np.exp(res.x) / np.sum(np.exp(res.x))
print("Achieved rho:", stats.spearmanr(dev_data.dot(w_opt), dev_target)[0])
print("Achieved r:", np.corrcoef(dev_data.dot(w_opt), dev_target)[0,1])

retr_cen_test = test_res[used_qpp_methods].apply(lambda x: np.log(1+x)).values
reader_cen_test = test_res[['perpC']].values
test_data = np.hstack((retr_cen_test, reader_cen_test))

print("TEST Achieved rho:", stats.spearmanr(test_data.dot(w_opt), test_res[target_metric])[0])
print("TEST Achieved r:", np.corrcoef(test_data.dot(w_opt), test_res[target_metric])[0,1])

Achieved rho: 0.23553112113194352
Achieved r: 0.23689852890526994
TEST Achieved rho: 0.2059935919352614
TEST Achieved r: 0.20775911726210847


In [29]:
coefs

array([ 0.02842572,  0.42649086,  0.11617598, -0.00117024,  0.15009278,
        0.00543349])

In [ ]:
w_opt 

#### Learned Combination -> Grid Search

In [7]:
def grid_search(_qpp_method):
    _rho_max = 0
    _coeff_lambda = 0.05
    for _c in np.union1d(-np.arange(0.00, 1.05, 0.050), np.arange(0.05, 1.05, 0.050)):
        _rho = stats.spearmanr((dev_res[_qpp_method].apply(lambda x: _c*(np.log(x+1)))+(1-_c)*dev_res.perpC).values, dev_res.f1.values)[0]
        if(_rho > _rho_max):
            _rho_max = _rho
            _coeff_lambda = _c
    print('accuracy on Dev set', _rho_max)
    return round(_coeff_lambda, 2)

def grid_search_then_test(_qpp_method):
    
    optimal_lambda = grid_search(_qpp_method)
    print(_qpp_method, 'optimal lambda is', optimal_lambda)
    predictions = (test_res[_qpp_method].apply(lambda x: optimal_lambda*(np.log(x+1)))+(1-optimal_lambda)*test_res.perpC).values
    rho = stats.spearmanr(predictions, test_res.f1.values)[0]
    print('accuracy on Test set', rho)
    rho = stats.spearmanr(predictions+test_res['prob(k)'].values, test_res.f1.values)[0]
    print('accuracy on Test set  (w/prob(k))', rho)

In [8]:
for qpp_method in ['nqc', 'maxScore', 'spatial', 'a_ratio', 'bertQPP']:
    print(qpp_method)
    grid_search_then_test(qpp_method)
    print('\n')

nqc
accuracy on Dev set 0.1858157877807126
nqc optimal lambda is 0.25
accuracy on Test set 0.15802850270490296
accuracy on Test set  (w/prob(k)) 0.24748990952561767


maxScore
accuracy on Dev set 0.2055670761350681
maxScore optimal lambda is 0.8
accuracy on Test set 0.16480413674824268
accuracy on Test set  (w/prob(k)) 0.29940338536161476


spatial
accuracy on Dev set 0.24236969183279303
spatial optimal lambda is 1.0
accuracy on Test set 0.22034333140646126
accuracy on Test set  (w/prob(k)) 0.33588569274138025


a_ratio
accuracy on Dev set 0.20784366403062318
a_ratio optimal lambda is 0.95
accuracy on Test set 0.17882425995683654
accuracy on Test set  (w/prob(k)) 0.3321868092276714


bertQPP
accuracy on Dev set 0.2414580282115419
bertQPP optimal lambda is 0.9
accuracy on Test set 0.2249304313105124
accuracy on Test set  (w/prob(k)) 0.3466498752071876


